In [1]:
print("Ok")

Ok


In [2]:
import sys
print(sys.executable)

c:\Users\HP\miniconda3\envs\medibot\python.exe


In [3]:
%pwd

'c:\\Users\\HP\\AI-ML-Workspace\\end-to-end-medical-chatbot\\research'

In [4]:
import os
os.chdir('../')

In [5]:
%pwd

'c:\\Users\\HP\\AI-ML-Workspace\\end-to-end-medical-chatbot'

In [6]:
import langchain
print(langchain.__version__)

1.3.11


In [7]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\HP\AppData\Local\Temp\ipykernel_4460\3754768683.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
c:\Users\HP\miniconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Extract data from pdf file
def load_pdf_file(data):
    loader = DirectoryLoader(data, 
                             glob="*.pdf", 
                             loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [9]:
extracted_data = load_pdf_file(data = 'Data/')

In [10]:
# extracted_data

In [11]:
# Split the data into text chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [12]:
text_chunks = text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 5859


In [13]:
# text_chunks

In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [15]:
# Download the embedding model from hugging face
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [16]:
embeddings = download_hugging_face_embeddings() 

C:\Users\HP\AppData\Local\Temp\ipykernel_4460\3007691038.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1909.11it/s]


In [17]:
query_result = embeddings.embed_query("Hello World")
print("Length", len(query_result))

Length 384


In [18]:
# query_result

In [62]:
from dotenv import load_dotenv
load_dotenv()

True

In [63]:
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
# OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

In [64]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medicalbot"

existing_indexes = [index.name for index in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created successfully!")
else:
    print("Index already exists!")

# Connect to the existing index
index = pc.Index(index_name)

Index already exists!


In [65]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
# os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [66]:
from langchain_pinecone import PineconeVectorStore

stats = index.describe_index_stats()

if stats["total_vector_count"] > 0:
    print("Index already contains vectors. Connecting...")
    docsearch = PineconeVectorStore(
        index=index,
        embedding=embeddings
    )
else:
    print("Index is empty. Uploading documents...")
    docsearch = PineconeVectorStore.from_documents(
        documents=text_chunks,
        embedding=embeddings,
        index_name=index_name
    )

Index already contains vectors. Connecting...


In [67]:
docsearch

In [68]:
retriever = docsearch.as_retriever(search_type = "similarity",search_kwargs={"k": 3})

In [69]:
retrieved_docs = retriever.invoke("What is acne?")

In [70]:
retrieved_docs

[Document(id='a8208171-3760-4693-9965-0497f6cae116', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='d6f145df-20f0-4a76-98d8-f2c32732a86a', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 38.0, 'page_label': '39', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM 

In [72]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # or another Groq-supported model
    temperature=0.4,
    max_tokens=500,
    api_key=GROQ_API_KEY
)

In [73]:
from langchain_classic.chains import LLMChain
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import RetrievalQAWithSourcesChain
from langchain_core.prompts import ChatPromptTemplate
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Don't try to make up an answer. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [74]:
question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=question_answer_chain)


In [75]:
response = rag_chain.invoke({"input": "What is acne?"})
print(response["answer"])

Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria. Acne vulgaris is the medical term for common acne, which is the most common skin disease.
